# 01 - Setup, Dataset Acquisition, Audit

**Purpose:** Inspect Colab resources, acquire Home Credit data (Kaggle API or manual upload), validate, convert to Parquet, audit, data dictionary, SQL analysis.

**Inputs:** Kaggle secrets (KAGGLE_USERNAME, KAGGLE_KEY) OR CSVs uploaded to data/raw/

**Outputs / persisted artifacts:** data/interim/*.parquet, artifacts/reports/{dataset_audit,relationship_checks,data_dictionary,leakage_audit}.csv, sql_*.csv

**Dependencies:** requirements.txt

**Execution order:** 1 of 6

**Validation checks:** File validation passes; 0 relationship violations; leakage audit lists every DAYS_* column

_All logic lives in `src/creditiq`; cells are safe to re-run (completed stages are skipped, artifacts live on Google Drive)._

In [ ]:
# --- Colab setup (safe to re-run) ---
import os, sys, subprocess
from pathlib import Path
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
ROOT = Path("/content/drive/MyDrive/CreditIQ") if Path("/content/drive/MyDrive").exists() else Path.cwd().parent
os.environ["CREDITIQ_ROOT"] = str(ROOT); sys.path.insert(0, str(ROOT / "src"))
if not (ROOT / "src").exists():
    raise SystemExit(f"Project not found at {ROOT}. Unzip CreditIQ into MyDrive/CreditIQ first.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=False)
from creditiq.config import get_paths, load_config, read_state
paths = get_paths(); cfg = load_config(paths)
print("Project root:", ROOT); print("Last completed stage:", read_state(paths).get("last_completed"))

In [ ]:
import psutil, shutil, subprocess
print(f"RAM available: {psutil.virtual_memory().available/1e9:.1f} GB | Disk free: {shutil.disk_usage('/content').free/1e9:.0f} GB | CPUs: {os.cpu_count()}")
print(subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout or "No GPU (not required: LightGBM/XGBoost run on CPU)")

**Dataset access.** Option A: add Colab Secrets `KAGGLE_USERNAME` / `KAGGLE_KEY` and accept the competition rules on Kaggle. Option B: upload the 8 CSVs to `MyDrive/CreditIQ/data/raw/`. Never paste tokens into cells.

In [ ]:
from creditiq import pipeline
pipeline.run("ingest")      # download (if needed) -> validate -> CSV to Parquet (resumable)
pipeline.run("audit")      # audit, relationships, data dictionary, leakage audit, SQL analysis queries

In [ ]:
import pandas as pd
display(pd.read_csv(paths.reports / "dataset_audit.csv")); display(pd.read_csv(paths.reports / "relationship_checks.csv"))
display(pd.read_csv(paths.reports / "leakage_audit.csv"))
for f in sorted(paths.reports.glob("sql_*.csv")): print(f.name); display(pd.read_csv(f))